# GéoMarketing IDF : J9 — Accessibilité aux transports en commun en Île-de-France

## Objectifs

- charger les arrêts et lignes IDFM ;
- normaliser les codes communaux ;
- regrouper les arrondissements de Paris ;
- distinguer bus, métro, tramway, RER et train ;
- éviter de compter plusieurs fois un même arrêt ;
- agréger les transports par commune ;
- joindre ces indicateurs au profil J8 ;
- produire le profil communal J9.

Ce notebook mesure la desserte structurelle des communes.

Il ne mesure pas encore :

- la fréquentation réelle des gares ;
- le nombre quotidien de voyageurs ;
- les temps de parcours ;
- la distance précise entre un restaurant et une gare ;
- les passages par heure.

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
import csv
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Bibliothèques chargées ✅")

Pandas : 2.2.2
NumPy : 1.26.4
Bibliothèques chargées ✅


In [8]:
#Définir les dossiers

RACINE = Path.cwd()

if not (RACINE / "data").exists():
    if (RACINE.parent / "data").exists():
        RACINE = RACINE.parent
    else:
        raise FileNotFoundError(
            "Le dossier data est introuvable. "
            "Ouvre Jupyter depuis la racine GeoMarketing_IDF "
            "ou depuis le dossier notebooks."
        )

DOSSIER_RAW_IDFM = RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF"/"data" / "raw" / "idfm"
DOSSIER_INTERIM = RACINE / "data" / "interim"
DOSSIER_PROCESSED = RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF/data" / "processed"

DOSSIER_RAW_IDFM.mkdir(parents=True, exist_ok=True)
DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

FICHIER_PROFIL_J8 = (
    DOSSIER_PROCESSED / "profil_communes_idf_j8.csv"
)

print("Racine :", RACINE)
print("Dossier IDFM :", DOSSIER_RAW_IDFM)
print("Profil J8 :", FICHIER_PROFIL_J8)

Racine : C:\Users\almou
Dossier IDFM : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\idfm
Profil J8 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j8.csv


In [9]:
#Afficher les fichiers disponibles dans le dossier IDFM

fichiers_idfm = sorted(
    fichier
    for fichier in DOSSIER_RAW_IDFM.rglob("*")
    if fichier.is_file()
)

if not fichiers_idfm:
    raise FileNotFoundError(
        "Aucun fichier trouvé dans data/raw/idfm."
    )

print("Fichiers disponibles :")

for fichier in fichiers_idfm:
    taille_mo = fichier.stat().st_size / 1_000_000

    print(
        f"- {fichier.relative_to(RACINE)} "
        f"({taille_mo:.2f} Mo)"
    )

Fichiers disponibles :
- OneDrive\GeoMarketing_IDF\data\raw\idfm\arrets_lignes_idfm.csv (13.92 Mo)


In [10]:
FICHIER_ARRETS_LIGNES = (
    DOSSIER_RAW_IDFM / "arrets_lignes_idfm.csv"
)

if not FICHIER_ARRETS_LIGNES.exists():
    candidats = [
        fichier
        for fichier in fichiers_idfm
        if fichier.suffix.lower() == ".csv"
        and "arret" in fichier.name.lower()
        and "ligne" in fichier.name.lower()
    ]

    if len(candidats) == 1:
        FICHIER_ARRETS_LIGNES = candidats[0]

    elif len(candidats) > 1:
        raise FileNotFoundError(
            "Plusieurs fichiers d'arrêts et lignes ont été trouvés. "
            "Indique manuellement le bon fichier dans "
            "FICHIER_ARRETS_LIGNES.\n"
            + "\n".join(str(fichier) for fichier in candidats)
        )

    else:
        raise FileNotFoundError(
            "Le fichier arrets-lignes.csv est introuvable dans "
            "data/raw/idfm."
        )

print("Fichier sélectionné :", FICHIER_ARRETS_LIGNES)
print(
    "Taille :",
    round(
        FICHIER_ARRETS_LIGNES.stat().st_size / 1_000_000,
        2,
    ),
    "Mo",
)

Fichier sélectionné : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\idfm\arrets_lignes_idfm.csv
Taille : 13.92 Mo


In [11]:
#Normaliser les noms de colonnes
def normaliser_nom_colonne(nom):
    nom = str(nom).strip().upper()

    nom = unicodedata.normalize("NFKD", nom)
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )

    nom = re.sub(r"[^A-Z0-9]+", "_", nom)

    return nom.strip("_")


def normaliser_texte(valeur):
    if pd.isna(valeur):
        return ""

    texte = str(valeur).strip().upper()

    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(
        caractere
        for caractere in texte
        if not unicodedata.combining(caractere)
    )

    texte = re.sub(r"[^A-Z0-9]+", " ", texte)
    texte = re.sub(r"\s+", " ", texte)

    return texte.strip()


def normaliser_code_commune(serie):
    resultat = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    valeurs_invalides = (
        resultat.isna()
        | resultat.str.lower().isin(
            ["", "nan", "none", "<na>"]
        )
    )

    resultat = resultat.mask(valeurs_invalides)

    valeurs_numeriques = resultat.str.fullmatch(
        r"\d{1,5}",
        na=False,
    )

    resultat = resultat.where(valeurs_numeriques)

    return resultat.str.zfill(5)


def convertir_nombre(serie):
    return pd.to_numeric(
        serie.astype(str)
        .str.replace("\u202f", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def detecter_separateur(fichier):
    with open(
        fichier,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline="",
    ) as flux:
        echantillon = flux.read(100_000)

    try:
        dialecte = csv.Sniffer().sniff(
            echantillon,
            delimiters=";,\t|",
        )

        return dialecte.delimiter

    except csv.Error:
        premiere_ligne = echantillon.splitlines()[0]

        separateurs = [";", ",", "\t", "|"]

        return max(
            separateurs,
            key=premiere_ligne.count,
        )


def concatener_valeurs_uniques(serie):
    valeurs = {
        str(valeur).strip()
        for valeur in serie.dropna()
        if str(valeur).strip()
    }

    return " | ".join(sorted(valeurs))


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "Enregistré :",
        fichier.relative_to(RACINE),
        f"— {len(table):,} lignes",
    )

In [12]:
#Charger le dataset J8

if not FICHIER_PROFIL_J8.exists():
    raise FileNotFoundError(
        "Le profil J8 est introuvable : "
        f"{FICHIER_PROFIL_J8}"
    )

profil_j8 = pd.read_csv(
    FICHIER_PROFIL_J8,
    dtype={"CODGEO": "string"},
    encoding="utf-8-sig",
    low_memory=False,
)

profil_j8.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j8.columns
]

if "CODGEO" not in profil_j8.columns:
    raise ValueError(
        "La colonne CODGEO est absente du profil J8."
    )

profil_j8["CODGEO"] = normaliser_code_commune(
    profil_j8["CODGEO"]
)

print("Dimensions du profil J8 :", profil_j8.shape)
print("Communes :", profil_j8["CODGEO"].nunique())

display(profil_j8.head())

Dimensions du profil J8 : (1266, 364)
Communes : 1266


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS,POP_FEMMES,POP_HOMMES,POP_F_0_2,POP_F_3_5,POP_F_6_10,POP_F_11_14,POP_F_15_17,POP_F_18_24,POP_F_25_39,POP_F_40_54,POP_F_55_64,POP_F_65_79,POP_F_80_PLUS,POP_H_0_2,POP_H_3_5,POP_H_6_10,POP_H_11_14,POP_H_15_17,POP_H_18_24,...,EMPLOIS_SALARIES_LT,EMPLOIS_SALARIES_FEMMES_LT,EMPLOIS_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_NON_SALARIES_LT,EMPLOIS_NON_SALARIES_FEMMES_LT,EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_LT_COMP,EMPLOIS_LT_AGRICULTEURS,EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS,EMPLOIS_LT_CADRES,EMPLOIS_LT_PROF_INTERMEDIAIRES,EMPLOIS_LT_EMPLOYES,EMPLOIS_LT_OUVRIERS,EMPLOIS_LT_AGRICULTURE,EMPLOIS_LT_INDUSTRIE,EMPLOIS_LT_CONSTRUCTION,EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES,EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL,EMPLOIS_FEMMES_LT_COMP,CHOMEURS_15_64_HOMMES,CHOMEURS_15_64_FEMMES,TAUX_ACTIVITE_15_64_PCT,TAUX_EMPLOI_15_64_PCT,TAUX_CHOMAGE_RP_15_64_PCT,TAUX_ACTIVITE_15_24_PCT,TAUX_EMPLOI_15_24_PCT,TAUX_CHOMAGE_RP_15_24_PCT,TAUX_ACTIVITE_25_54_PCT,TAUX_EMPLOI_25_54_PCT,TAUX_CHOMAGE_RP_25_54_PCT,TAUX_ACTIVITE_55_64_PCT,TAUX_EMPLOI_55_64_PCT,TAUX_CHOMAGE_RP_55_64_PCT,TAUX_ACTIVITE_15_64_HOMMES_PCT,TAUX_EMPLOI_15_64_HOMMES_PCT,TAUX_CHOMAGE_RP_15_64_HOMMES_PCT,TAUX_ACTIVITE_15_64_FEMMES_PCT,TAUX_EMPLOI_15_64_FEMMES_PCT,TAUX_CHOMAGE_RP_15_64_FEMMES_PCT,ECART_TAUX_EMPLOI_FEMMES_HOMMES_POINTS,PART_INACTIFS_15_64_PCT,PART_ELEVES_ETUDIANTS_STAGIAIRES_15_64_PCT,PART_RETRAITES_PRERETRAITES_15_64_PCT,PART_AUTRES_INACTIFS_15_64_PCT,INDICE_CONCENTRATION_EMPLOI,ECART_EMPLOIS_LT_ACTIFS_RESIDENTS,EMPLOIS_TEMPS_PARTIEL_LT,PART_EMPLOIS_SALARIES_LT_PCT,PART_EMPLOIS_NON_SALARIES_LT_PCT,PART_EMPLOIS_TEMPS_PARTIEL_LT_PCT,PART_EMPLOIS_FEMMES_LT_PCT,PART_EMPLOIS_LT_AGRICULTEURS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS_PCT,PART_EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS_PCT,PART_EMPLOIS_LT_CADRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_CADRES_PCT,PART_EMPLOIS_LT_PROF_INTERMEDIAIRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES_PCT,PART_EMPLOIS_LT_EMPLOYES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_EMPLOYES_PCT,PART_EMPLOIS_LT_OUVRIERS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_OUVRIERS_PCT,PART_EMPLOIS_LT_AGRICULTURE_PCT,PART_EMPLOIS_LT_INDUSTRIE_PCT,PART_EMPLOIS_LT_CONSTRUCTION_PCT,PART_EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES_PCT,PART_EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL_PCT,EMPLOIS_TERTIAIRES_LT,PART_EMPLOIS_TERTIAIRES_LT_PCT,EMPLOIS_15P_LT_POUR_1000_HAB,INDICE_DENSITE_EMPLOIS_IDF_BASE100,NB_RESTAURATION_RAPIDE_1000_EMPLOIS,INDICE_DENSITE_RAPIDE_PAR_EMPLOI_IDF_BASE100,NB_RESTAURANTS_TOTAL_1000_EMPLOIS
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508,513386.756029,457511.658564,387099.215350,302819.494109,154040.332162,24566.982278,787667.317537,970898.414593,481426.808549,178607.314440,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.50,-0.59,33650.0,16.8,1.551134e+06,188002.0,73.384590,88.944294,20911,8329,1

In [13]:
#Charger le fichier des transports IDFM
separateur_idfm = detecter_separateur(
    FICHIER_ARRETS_LIGNES
)

print("Séparateur détecté :", repr(separateur_idfm))

transport_brut = pd.read_csv(
    FICHIER_ARRETS_LIGNES,
    sep=separateur_idfm,
    dtype=str,
    encoding="utf-8-sig",
    encoding_errors="replace",
    low_memory=False,
)

transport_brut.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in transport_brut.columns
]

print("Dimensions :", transport_brut.shape)
print("Colonnes IDFM :")
print(transport_brut.columns.tolist())

display(transport_brut.head())

Séparateur détecté : ';'
Dimensions : (74959, 13)
Colonnes IDFM :
['ID', 'ROUTE_LONG_NAME', 'STOP_ID', 'STOP_NAME', 'STOP_LON', 'STOP_LAT', 'OPERATORNAME', 'SHORTNAME', 'BOOKINGRULES', 'MODE', 'POINTGEO', 'NOM_COMMUNE', 'CODE_INSEE']


,ID,ROUTE_LONG_NAME,STOP_ID,STOP_NAME,STOP_LON,STOP_LAT,OPERATORNAME,SHORTNAME,BOOKINGRULES,MODE,POINTGEO,NOM_COMMUNE,CODE_INSEE
0,IDFM:C01389,T1,IDFM:23293,Chemin des Reniers,2.321527043611886,48.934560806805955,RATP,T1,NaN,Tramway,"48.934560806805955, 2.321527043611886",Villeneuve-la-Garenne,92078
1,IDFM:C01389,T1,IDFM:23313,Théâtre Gérard Philipe,2.350468785750953,48.93749994624904,RATP,T1,NaN,Tramway,"48.93749994624904, 2.350468785750953",Saint-Denis,93066
2,IDFM:C01389,T1,IDFM:22268,Marché de Saint-Denis,2.35587486964062,48.93856329762427,RATP,T1,NaN,Tramway,"48.93856329762427, 2.35587486964062",Saint-Denis,93066
3,IDFM:C01389,T1,IDFM:24429,Stade Géo André,2.4020687684654924,48.92433314315498,RATP,T1,NaN,Tramway,"48.92433314315498, 2.4020687684654924",La Courneuve,93027
4,IDFM:C01389,T1,IDFM:24427,Danton,2.406618691605552,48.922656100102074,RATP,T1,NaN,Tramway,"48.922656100102074, 2.406618691605552",La Courneuve,93027


In [14]:
#Identifier automatiquement les colonnes

alias_colonnes = {
    "ROUTE_ID": [
        "ROUTE_ID",
        "ID_LIGNE",
        "LINE_ID",
        "ID",
    ],
    "ROUTE_LONG_NAME": [
        "ROUTE_LONG_NAME",
        "NOM_LONG_LIGNE",
        "NOM_LIGNE",
        "LIGNE",
    ],
    "ROUTE_SHORT_NAME": [
        "ROUTE_SHORT_NAME",
        "SHORTNAME",
        "NOM_COURT_LIGNE",
        "NUMERO_LIGNE",
    ],
    "STOP_ID": [
        "STOP_ID",
        "ID_ARRET",
        "ID_REF_ARRET",
        "ID_REF_A",
    ],
    "STOP_NAME": [
        "STOP_NAME",
        "NOM_ARRET",
        "LIBELLE_ARRET",
        "NOM",
    ],
    "STOP_LON": [
        "STOP_LON",
        "LONGITUDE",
        "LON",
    ],
    "STOP_LAT": [
        "STOP_LAT",
        "LATITUDE",
        "LAT",
    ],
    "OPERATOR_NAME": [
        "OPERATOR_NAME",
        "OPERATORNAME",
        "NOM_OPERATEUR",
        "AGENCY_NAME",
    ],
    "MODE_SOURCE": [
        "MODE_SOURCE",
        "MODE",
        "ROUTE_TYPE",
        "TYPE_MODE",
    ],
    "NOM_COMMUNE_SOURCE": [
        "NOM_COMMUNE_SOURCE",
        "NOM_COMMUNE",
        "COMMUNE",
        "CITY",
    ],
    "CODGEO": [
        "CODGEO",
        "CODE_INSEE",
        "INSEE_COM",
        "CODE_COMMUNE",
        "DEPCOM",
    ],
}


def trouver_colonne(table, candidats):
    return next(
        (
            candidat
            for candidat in candidats
            if candidat in table.columns
        ),
        None,
    )


correspondance_colonnes = {}

for colonne_cible, candidats in alias_colonnes.items():
    colonne_source = trouver_colonne(
        transport_brut,
        candidats,
    )

    correspondance_colonnes[colonne_cible] = (
        colonne_source
    )

controle_colonnes = pd.DataFrame(
    {
        "COLONNE_CIBLE": correspondance_colonnes.keys(),
        "COLONNE_SOURCE": correspondance_colonnes.values(),
    }
)

display(controle_colonnes)

,COLONNE_CIBLE,COLONNE_SOURCE
0,ROUTE_ID,ID
1,ROUTE_LONG_NAME,ROUTE_LONG_NAME
2,ROUTE_SHORT_NAME,SHORTNAME
3,STOP_ID,STOP_ID
4,STOP_NAME,STOP_NAME
5,STOP_LON,STOP_LON
6,STOP_LAT,STOP_LAT
7,OPERATOR_NAME,OPERATORNAME
8,MODE_SOURCE,MODE
9,NOM_COMMUNE_SOURCE,NOM_COMMUNE


In [15]:
#Renommer et sélectionner les colonnes

renommage = {
    colonne_source: colonne_cible
    for colonne_cible, colonne_source
    in correspondance_colonnes.items()
    if colonne_source is not None
}

arrets_lignes = transport_brut.rename(
    columns=renommage
).copy()

colonnes_optionnelles = [
    "ROUTE_LONG_NAME",
    "ROUTE_SHORT_NAME",
    "OPERATOR_NAME",
    "NOM_COMMUNE_SOURCE",
]

for colonne in colonnes_optionnelles:
    if colonne not in arrets_lignes.columns:
        arrets_lignes[colonne] = pd.NA

colonnes_conservees = [
    "ROUTE_ID",
    "ROUTE_LONG_NAME",
    "ROUTE_SHORT_NAME",
    "STOP_ID",
    "STOP_NAME",
    "STOP_LON",
    "STOP_LAT",
    "OPERATOR_NAME",
    "MODE_SOURCE",
    "NOM_COMMUNE_SOURCE",
    "CODGEO",
]

arrets_lignes = arrets_lignes[
    colonnes_conservees
].copy()

print("Dimensions après sélection :", arrets_lignes.shape)

display(arrets_lignes.head())

Dimensions après sélection : (74959, 11)


,ROUTE_ID,ROUTE_LONG_NAME,ROUTE_SHORT_NAME,STOP_ID,STOP_NAME,STOP_LON,STOP_LAT,OPERATOR_NAME,MODE_SOURCE,NOM_COMMUNE_SOURCE,CODGEO
0,IDFM:C01389,T1,T1,IDFM:23293,Chemin des Reniers,2.321527043611886,48.934560806805955,RATP,Tramway,Villeneuve-la-Garenne,92078
1,IDFM:C01389,T1,T1,IDFM:23313,Théâtre Gérard Philipe,2.350468785750953,48.93749994624904,RATP,Tramway,Saint-Denis,93066
2,IDFM:C01389,T1,T1,IDFM:22268,Marché de Saint-Denis,2.35587486964062,48.93856329762427,RATP,Tramway,Saint-Denis,93066
3,IDFM:C01389,T1,T1,IDFM:24429,Stade Géo André,2.4020687684654924,48.92433314315498,RATP,Tramway,La Courneuve,93027
4,IDFM:C01389,T1,T1,IDFM:24427,Danton,2.406618691605552,48.922656100102074,RATP,Tramway,La Courneuve,93027


In [16]:
#Nettoyer les variables
colonnes_texte = [
    "ROUTE_ID",
    "ROUTE_LONG_NAME",
    "ROUTE_SHORT_NAME",
    "STOP_ID",
    "STOP_NAME",
    "OPERATOR_NAME",
    "MODE_SOURCE",
    "NOM_COMMUNE_SOURCE",
]

for colonne in colonnes_texte:
    arrets_lignes[colonne] = (
        arrets_lignes[colonne]
        .astype("string")
        .str.strip()
    )

    valeurs_invalides = (
        arrets_lignes[colonne].isna()
        | arrets_lignes[colonne]
        .str.lower()
        .isin(["", "nan", "none", "<na>"])
    )

    arrets_lignes[colonne] = (
        arrets_lignes[colonne]
        .mask(valeurs_invalides)
    )

arrets_lignes["CODGEO_SOURCE"] = (
    normaliser_code_commune(
        arrets_lignes["CODGEO"]
    )
)

arrets_lignes["CODGEO"] = (
    arrets_lignes["CODGEO_SOURCE"]
)

arrets_lignes["STOP_LAT"] = convertir_nombre(
    arrets_lignes["STOP_LAT"]
)

arrets_lignes["STOP_LON"] = convertir_nombre(
    arrets_lignes["STOP_LON"]
)

arrets_lignes["COORDONNEES_VALIDES"] = (
    arrets_lignes["STOP_LAT"].between(48.0, 49.3)
    & arrets_lignes["STOP_LON"].between(1.4, 3.7)
)

print(
    "Coordonnées plausibles :",
    arrets_lignes["COORDONNEES_VALIDES"].sum(),
)

print(
    "Coordonnées absentes ou anormales :",
    (~arrets_lignes["COORDONNEES_VALIDES"]).sum(),
)

Coordonnées plausibles : 74919
Coordonnées absentes ou anormales : 40


In [17]:
#Harmoniser les codes d'arrondissements et postaux de Saint-Denis

codes_profil = set(
    profil_j8["CODGEO"].dropna()
)

nombre_paris_avant = 0

if "75056" in codes_profil:
    masque_arrondissements_paris = (
        arrets_lignes["CODGEO"]
        .str.fullmatch(
            r"751(0[1-9]|1[0-9]|20)",
            na=False,
        )
    )

    nombre_paris_avant = (
        masque_arrondissements_paris.sum()
    )

    arrets_lignes.loc[
        masque_arrondissements_paris,
        "CODGEO",
    ] = "75056"

if (
    "93066" in codes_profil
    and "93059" not in codes_profil
):
    masque_pierrefitte = (
        arrets_lignes["CODGEO"].eq("93059")
    )

    nombre_pierrefitte = masque_pierrefitte.sum()

    arrets_lignes.loc[
        masque_pierrefitte,
        "CODGEO",
    ] = "93066"

else:
    nombre_pierrefitte = 0

print(
    "Lignes d'arrêts des arrondissements de Paris "
    "regroupées sous 75056 :",
    nombre_paris_avant,
)

print(
    "Lignes de Pierrefitte regroupées sous 93066 :",
    nombre_pierrefitte,
)

Lignes d'arrêts des arrondissements de Paris regroupées sous 75056 : 0
Lignes de Pierrefitte regroupées sous 93066 : 0


In [18]:
#Filtrer sur l'Ile-de-France

DEPARTEMENTS_IDF = {
    "75",
    "77",
    "78",
    "91",
    "92",
    "93",
    "94",
    "95",
}

masque_code_idf = (
    arrets_lignes["CODGEO"]
    .str.fullmatch(r"\d{5}", na=False)
    & arrets_lignes["CODGEO"]
    .str[:2]
    .isin(DEPARTEMENTS_IDF)
)

arrets_lignes_idf = arrets_lignes[
    masque_code_idf
].copy()

arrets_lignes_idf = arrets_lignes_idf.dropna(
    subset=[
        "CODGEO",
        "ROUTE_ID",
        "STOP_ID",
        "STOP_NAME",
        "MODE_SOURCE",
    ]
)

nombre_avant_dedoublonnage = len(
    arrets_lignes_idf
)

arrets_lignes_idf = (
    arrets_lignes_idf
    .drop_duplicates(
        subset=[
            "CODGEO",
            "STOP_ID",
            "ROUTE_ID",
        ]
    )
    .copy()
)

print(
    "Lignes conservées :",
    len(arrets_lignes_idf),
)

print(
    "Doublons arrêt-ligne supprimés :",
    nombre_avant_dedoublonnage
    - len(arrets_lignes_idf),
)

print(
    "Codes communaux représentés :",
    arrets_lignes_idf["CODGEO"].nunique(),
)

Lignes conservées : 74674
Doublons arrêt-ligne supprimés : 0
Codes communaux représentés : 1261


In [19]:
#Analyser les modes de transport d'origine
controle_modes_source = (
    arrets_lignes_idf["MODE_SOURCE"]
    .value_counts(dropna=False)
    .rename_axis("MODE_SOURCE")
    .reset_index(name="NOMBRE")
)

display(controle_modes_source)

,MODE_SOURCE,NOMBRE
0,Bus,72712
1,Metro,803
2,Tramway,564
3,RapidTransit,248
4,LocalTrain,235
5,regionalRail,82
6,RailShuttle,16
7,CableWay,10
8,Funicular,4


In [20]:
#Classer les modes de transport

def classer_mode_transport(ligne):
    mode_brut = ligne["MODE_SOURCE"]
    nom_court = normaliser_texte(
        ligne["ROUTE_SHORT_NAME"]
    )
    nom_long = normaliser_texte(
        ligne["ROUTE_LONG_NAME"]
    )

    texte_mode = normaliser_texte(mode_brut)
    texte_complet = " ".join(
        [texte_mode, nom_court, nom_long]
    )

    mode_numerique = None

    try:
        mode_numerique = int(
            float(
                str(mode_brut)
                .strip()
                .replace(",", ".")
            )
        )
    except (TypeError, ValueError):
        pass

    mode_base = None

    if mode_numerique is not None:
        if mode_numerique == 0:
            mode_base = "TRAMWAY"
        elif mode_numerique == 1:
            mode_base = "METRO"
        elif mode_numerique == 2:
            mode_base = "TRAIN"
        elif mode_numerique == 3:
            mode_base = "BUS"
        elif mode_numerique == 7:
            mode_base = "FUNICULAIRE"
        elif 100 <= mode_numerique <= 199:
            mode_base = "TRAIN"
        elif 400 <= mode_numerique <= 499:
            mode_base = "METRO"
        elif 700 <= mode_numerique <= 799:
            mode_base = "BUS"
        elif 900 <= mode_numerique <= 999:
            mode_base = "TRAMWAY"

    if mode_base is None:
        if "RER" in texte_complet:
            mode_base = "RER"

        elif "METRO" in texte_complet:
            mode_base = "METRO"

        elif "TRAM" in texte_complet:
            mode_base = "TRAMWAY"

        elif "FUNICULAIRE" in texte_complet:
            mode_base = "FUNICULAIRE"

        elif (
            "BUS" in texte_complet
            or "AUTOCAR" in texte_complet
            or "COACH" in texte_complet
        ):
            mode_base = "BUS"

        elif (
            "TRAIN" in texte_complet
            or "RAIL" in texte_complet
            or "TRANSILIEN" in texte_complet
            or "TER" in texte_complet
        ):
            mode_base = "TRAIN"

        else:
            mode_base = "AUTRE"

    # Les lignes A à E du mode ferroviaire sont les RER.
    if (
        mode_base == "TRAIN"
        and nom_court in {"A", "B", "C", "D", "E"}
    ):
        mode_base = "RER"

    return mode_base


arrets_lignes_idf["MODE"] = (
    arrets_lignes_idf.apply(
        classer_mode_transport,
        axis=1,
    )
)

controle_modes = pd.crosstab(
    arrets_lignes_idf["MODE_SOURCE"],
    arrets_lignes_idf["MODE"],
)

display(controle_modes)

MODE,AUTRE,BUS,FUNICULAIRE,METRO,RER,TRAIN,TRAMWAY
MODE_SOURCE,,,,,,,
Bus,0,72337,0,43,213,0,119
CableWay,10,0,0,0,0,0,0
Funicular,0,0,4,0,0,0,0
LocalTrain,0,0,0,0,0,235,0
Metro,0,0,0,803,0,0,0
RailShuttle,0,0,0,0,0,16,0
RapidTransit,248,0,0,0,0,0,0
Tramway,0,0,0,0,0,0,564
regionalRail,0,0,0,0,0,82,0


In [21]:
#Contrôler les modes de transport non classés

modes_non_classes = arrets_lignes_idf[
    arrets_lignes_idf["MODE"].eq("AUTRE")
][
    [
        "MODE_SOURCE",
        "ROUTE_SHORT_NAME",
        "ROUTE_LONG_NAME",
    ]
].drop_duplicates()

print(
    "Lignes arrêt-ligne classées AUTRE :",
    arrets_lignes_idf["MODE"]
    .eq("AUTRE")
    .sum(),
)

display(modes_non_classes.head(50))

Lignes arrêt-ligne classées AUTRE : 258


,MODE_SOURCE,ROUTE_SHORT_NAME,ROUTE_LONG_NAME
11735,RapidTransit,C,C
11755,RapidTransit,D,D
11769,RapidTransit,E,E
11775,RapidTransit,A,A
11936,RapidTransit,B,B
11946,CableWay,C1,C1


In [23]:
#Construire les lieux et noms d'arret

nom_ligne_court = (
    arrets_lignes_idf["ROUTE_SHORT_NAME"]
    .astype("string")
    .str.strip()
)

nom_ligne_court = nom_ligne_court.mask(
    nom_ligne_court.eq("")
)

nom_ligne_long = (
    arrets_lignes_idf["ROUTE_LONG_NAME"]
    .astype("string")
    .str.strip()
)

nom_ligne_long = nom_ligne_long.mask(
    nom_ligne_long.eq("")
)

arrets_lignes_idf["NOM_LIGNE"] = (
    nom_ligne_court
    .fillna(nom_ligne_long)
    .fillna(arrets_lignes_idf["ROUTE_ID"])
)

arrets_lignes_idf["NOM_ARRET_NORMALISE"] = (
    arrets_lignes_idf["STOP_NAME"]
    .map(normaliser_texte)
)

masque_nom_vide = (
    arrets_lignes_idf["NOM_ARRET_NORMALISE"]
    .eq("")
)

arrets_lignes_idf.loc[
    masque_nom_vide,
    "NOM_ARRET_NORMALISE",
] = arrets_lignes_idf.loc[
    masque_nom_vide,
    "STOP_ID",
].map(normaliser_texte)

arrets_lignes_idf["CLE_LIEU"] = (
    arrets_lignes_idf["CODGEO"]
    + " | "
    + arrets_lignes_idf["NOM_ARRET_NORMALISE"]
)

MODES_FERRES = {
    "METRO",
    "TRAMWAY",
    "RER",
    "TRAIN",
    "FUNICULAIRE",
}

MODES_LOURDS = {
    "METRO",
    "RER",
    "TRAIN",
}

for mode in [
    "BUS",
    "METRO",
    "TRAMWAY",
    "RER",
    "TRAIN",
    "FUNICULAIRE",
    "AUTRE",
]:
    arrets_lignes_idf[f"EST_{mode}"] = (
        arrets_lignes_idf["MODE"].eq(mode)
    )

arrets_lignes_idf["EST_FERRE"] = (
    arrets_lignes_idf["MODE"]
    .isin(MODES_FERRES)
)

arrets_lignes_idf["EST_LOURD"] = (
    arrets_lignes_idf["MODE"]
    .isin(MODES_LOURDS)
)

display(
    arrets_lignes_idf[
        [
            "CODGEO",
            "STOP_NAME",
            "STOP_ID",
            "ROUTE_ID",
            "NOM_LIGNE",
            "MODE",
            "CLE_LIEU",
        ]
    ].head()
)

,CODGEO,STOP_NAME,STOP_ID,ROUTE_ID,NOM_LIGNE,MODE,CLE_LIEU
0,92078,Chemin des Reniers,IDFM:23293,IDFM:C01389,T1,TRAMWAY,92078 | CHEMIN DES RENIERS
1,93066,Théâtre Gérard Philipe,IDFM:23313,IDFM:C01389,T1,TRAMWAY,93066 | THEATRE GERARD PHILIPE
2,93066,Marché de Saint-Denis,IDFM:22268,IDFM:C01389,T1,TRAMWAY,93066 | MARCHE DE SAINT DENIS
3,93027,Stade Géo André,IDFM:24429,IDFM:C01389,T1,TRAMWAY,93027 | STADE GEO ANDRE
4,93027,Danton,IDFM:24427,IDFM:C01389,T1,TRAMWAY,93027 | DANTON


In [26]:
arrets_lignes_idf[arrets_lignes_idf["EST_RER"]]

,ROUTE_ID,ROUTE_LONG_NAME,ROUTE_SHORT_NAME,STOP_ID,STOP_NAME,STOP_LON,STOP_LAT,OPERATOR_NAME,MODE_SOURCE,NOM_COMMUNE_SOURCE,CODGEO,CODGEO_SOURCE,COORDONNEES_VALIDES,MODE,NOM_LIGNE,NOM_ARRET_NORMALISE,CLE_LIEU,EST_BUS,EST_METRO,EST_TRAMWAY,EST_RER,EST_TRAIN,EST_FUNICULAIRE,EST_AUTRE,EST_FERRE,EST_LOURD
7427,IDFM:C01849,Remplacement RER A,A,IDFM:8917,Gare de Cergy Saint-Christophe,2.033868,49.049844,SNCF,Bus,Cergy,95127,95127,True,RER,A,GARE DE CERGY SAINT CHRISTOPHE,95127 | GARE DE CERGY SAINT CHRISTOPHE,False,False,False,True,False,False,False,True,True
7428,IDFM:C01849,Remplacement RER A,A,IDFM:493394,Gare de Cergy Préfecture - Quai G,2.080554,49.036263,SNCF,Bus,Cergy,95127,95127,True,RER,A,GARE DE CERGY PREFECTURE QUAI G,95127 | GARE DE CERGY PREFECTURE QUAI G,False,False,False,True,False,False,False,True,True
7429,IDFM:C01849,Remplacement RER A,A,IDFM:28984,Gare Saint-Lazare,2.326020,48.875411,SNCF,Bus,Paris,75056,75056,True,RER,A,GARE SAINT LAZARE,75056 | GARE SAINT LAZARE,False,False,False,True,False,False,False,True,True
7430,IDFM:C01850,Remplacement RER B,B,IDFM:36616,Gare du Parc des Expositions,2.513309,48.973496,SNCF,Bus,Villepinte,93078,93078,True,RER,B,GARE DU PARC DES EXPOSITIONS,93078 | GARE DU PARC DES EXPOSITIONS,False,False,False,True,False,False,False,True,True
7431,IDFM:C01850,Remplacement RER B,B,IDFM:474234,Gare de Sevran Beaudottes,2.523797,48.946974,SNCF,Bus,Sevran,93071,93071,True,RER,B,GARE DE SEVRAN BEAUDOTTES,93071 | GARE DE SEVRAN BEAUDOTTES,False,False,False,True,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68636,IDFM:C01852,Remplacement RER D,D,IDFM:17378,Gare de Montgeron - Crosne,2.462481,48.707942,SNCF,Bus,Montgeron,91421,91421,True,RER,D,GARE DE MONTGERON CROSNE,91421 | GARE DE MONTGERON CROSNE,False,False,False,True,False,False,False,True,True
68637,IDFM:C01853,Remplacement RER E,E,IDFM:478447,Les Yvris - Noisy-le-Grand RER,2.582169,48.823779,SNCF,Bus,Noisy-le-Grand,93051,93051,True,RER,E,LES YVRIS NOISY LE GRAND RER,93051 | LES YVRIS NOISY LE GRAND RER,False,False,False,True,False,False,False,True,True
68638,IDFM:C01853,Remplacement RER E,E,IDFM:36432,Gagny RER,2.525748,48.882655,SNCF,Bus,Villemomble,93077,93077,True,RER,E,GAGNY RER,93077 | GAGNY RER,False,False,False,True,False,False,False,True,True
68639,IDFM:C01853,Remplacement RER E,E,IDFM:26157,Noisy-le-Sec RER,2.460962,48.896024,SNCF,Bus,Noisy-le-Sec,93053,93053,True,RER,E,NOISY LE SEC RER,93053 | NOISY LE SEC RER,False,False,False,True,False,False,False,True,True


In [27]:
#Créer la table des lieux de transport

lieux_transport = (
    arrets_lignes_idf
    .groupby(
        ["CODGEO", "CLE_LIEU"],
        as_index=False,
    )
    .agg(
        NOM_ARRET=("STOP_NAME", "first"),
        NOM_COMMUNE_SOURCE=(
            "NOM_COMMUNE_SOURCE",
            "first",
        ),
        LATITUDE=("STOP_LAT", "median"),
        LONGITUDE=("STOP_LON", "median"),
        NB_IDENTIFIANTS_ARRET=(
            "STOP_ID",
            "nunique",
        ),
        NB_LIGNES=("ROUTE_ID", "nunique"),
        NB_MODES=("MODE", "nunique"),
        NB_OPERATEURS=(
            "OPERATOR_NAME",
            "nunique",
        ),
        MODES=(
            "MODE",
            concatener_valeurs_uniques,
        ),
        LIGNES=(
            "NOM_LIGNE",
            concatener_valeurs_uniques,
        ),
        OPERATEURS=(
            "OPERATOR_NAME",
            concatener_valeurs_uniques,
        ),
        A_BUS=("EST_BUS", "max"),
        A_METRO=("EST_METRO", "max"),
        A_TRAMWAY=("EST_TRAMWAY", "max"),
        A_RER=("EST_RER", "max"),
        A_TRAIN=("EST_TRAIN", "max"),
        A_FUNICULAIRE=(
            "EST_FUNICULAIRE",
            "max",
        ),
        A_AUTRE=("EST_AUTRE", "max"),
        A_FERRE=("EST_FERRE", "max"),
        A_LOURD=("EST_LOURD", "max"),
    )
)

lieux_transport["EST_MULTIMODAL"] = (
    lieux_transport["NB_MODES"] >= 2
)

lieux_transport[
    "EST_POLE_MULTIMODAL_LOURD"
] = (
    lieux_transport["EST_MULTIMODAL"]
    & lieux_transport["A_LOURD"]
)

print(
    "Nombre de lieux de transport :",
    len(lieux_transport),
)

print(
    "Lieux multimodaux :",
    lieux_transport["EST_MULTIMODAL"].sum(),
)

print(
    "Pôles multimodaux lourds :",
    lieux_transport[
        "EST_POLE_MULTIMODAL_LOURD"
    ].sum(),
)

display(lieux_transport.head())

Nombre de lieux de transport : 18822
Lieux multimodaux : 631
Pôles multimodaux lourds : 440


,CODGEO,CLE_LIEU,NOM_ARRET,NOM_COMMUNE_SOURCE,LATITUDE,LONGITUDE,NB_IDENTIFIANTS_ARRET,NB_LIGNES,NB_MODES,NB_OPERATEURS,MODES,LIGNES,OPERATEURS,A_BUS,A_METRO,A_TRAMWAY,A_RER,A_TRAIN,A_FUNICULAIRE,A_AUTRE,A_FERRE,A_LOURD,EST_MULTIMODAL,EST_POLE_MULTIMODAL_LOURD
0,75056,75056 | 88 RUE LEPIC,"88, Rue Lepic",Paris,48.887313,2.336285,1,1,1,1,BUS,40,RATP,True,False,False,False,False,False,False,False,False,False,False
1,75056,75056 | ABBE GEORGES HENOCQUE,Abbé Georges Henocque,Paris,48.823768,2.353282,2,1,1,1,BUS,57,RATP,True,False,False,False,False,False,False,False,False,False,False
2,75056,75056 | ABBE GROULT,Abbé Groult,Paris,48.839634,2.297632,2,4,1,2,BUS,39 | 80 | N13 | N62,ATM Croix du Sud | RATP,True,False,False,False,False,False,False,False,False,False,False
3,75056,75056 | ABBESSES,Abbesses,Paris,48.884388,2.338381,4,2,2,1,BUS | METRO,12 | 40,RATP,True,True,False,False,False,False,False,True,True,True,True
4,75056,75056 | ABEILLE,Abeille,Paris,48.901500,2.361913,1,1,1,1,BUS,NEY-FLA,RATP,True,False,False,False,False,False,False,False,False,False,False


In [28]:
#Contrôler les principaux pôles

principaux_poles = (
    lieux_transport
    .sort_values(
        [
            "NB_MODES",
            "NB_LIGNES",
            "NB_IDENTIFIANTS_ARRET",
        ],
        ascending=False,
    )
    [
        [
            "CODGEO",
            "NOM_ARRET",
            "MODES",
            "NB_MODES",
            "NB_LIGNES",
            "LIGNES",
            "LATITUDE",
            "LONGITUDE",
        ]
    ]
    .head(30)
)

display(principaux_poles)

,CODGEO,NOM_ARRET,MODES,NB_MODES,NB_LIGNES,LIGNES,LATITUDE,LONGITUDE
418,75056,Gare du Nord,AUTRE | BUS | METRO | RER | TRAIN,5,24,302 | 35 | 38 | 39 | 4 | 43 | 45 | 48 | 5 | 91 | B | D | H | K | N140 | N143 | N147 | N148 | N43 | TER,48.880981,2.357555
405,75056,Gare d'Austerlitz,AUTRE | BUS | METRO | RER | TRAIN,5,16,10 | 215 | 24 | 5 | 57 | 63 | 89 | 91 | C | N01 | N02 | N131 | N133 | N31 | TER,48.843396,2.364586
13758,93008,Bobigny - Pablo Picasso,BUS | METRO | RER | TRAMWAY,4,16,146 | 148 | 234 | 251 | 301 | 322 | 5 | 615 | 620 | E | EX 93 | N13 | N41 | N45 | T1,48.907238,2.449413
413,75056,Gare de Lyon,AUTRE | BUS | METRO | TRAIN,4,13,1 | 14 | 24 | 63 | 72 | 77 | 87 | A | D | N32 | N35 | R | TER,48.843885,2.373658
910,75056,Pont Cardinet,BUS | METRO | RER | TRAIN,4,10,14 | 163 | 28 | 31 | 94 | A | J | L | N154,48.887544,2.313375
94,75056,Bibliothèque François Mitterrand,AUTRE | BUS | METRO | RER,4,9,132 | 14 | 25 | 325 | 62 | 71 | 89 | C,48.830045,2.377197
1098,75056,Rosa Parks,AUTRE | BUS | RER | TRAMWAY,4,6,239 | 60 | E | N140 | T3b,48.896146,2.373724
13110,92062,La Défense,AUTRE | METRO | TRAIN | TRAMWAY,4,6,1 | A | E | L | T2 | U,48.892230,2.237976
5427,78005,Achères Ville,AUTRE | BUS | RER | TRAIN,4,6,1222 | A | L | N152,48.969972,2.077342
13886,93013,Le Bourget,AUTRE | BUS | RER | TRAMWAY,4,4,133 | B | T11,48.930744,2.424269


In [29]:
#Agréger les transports par commune

lieux_communes = (
    lieux_transport
    .groupby("CODGEO", as_index=False)
    .agg(
        NB_LIEUX_TRANSPORT=(
            "CLE_LIEU",
            "nunique",
        ),
        NB_LIEUX_BUS=("A_BUS", "sum"),
        NB_STATIONS_METRO=("A_METRO", "sum"),
        NB_STATIONS_TRAMWAY=(
            "A_TRAMWAY",
            "sum",
        ),
        NB_GARES_RER=("A_RER", "sum"),
        NB_GARES_TRAIN=("A_TRAIN", "sum"),
        NB_STATIONS_FUNICULAIRE=(
            "A_FUNICULAIRE",
            "sum",
        ),
        NB_LIEUX_FERRES=("A_FERRE", "sum"),
        NB_GARES_STATIONS_TRANSPORT_LOURD=(
            "A_LOURD",
            "sum",
        ),
        NB_LIEUX_MULTIMODAUX=(
            "EST_MULTIMODAL",
            "sum",
        ),
        NB_POLES_MULTIMODAUX_LOURDS=(
            "EST_POLE_MULTIMODAL_LOURD",
            "sum",
        ),
    )
)

display(lieux_communes.head())

,CODGEO,NB_LIEUX_TRANSPORT,NB_LIEUX_BUS,NB_STATIONS_METRO,NB_STATIONS_TRAMWAY,NB_GARES_RER,NB_GARES_TRAIN,NB_STATIONS_FUNICULAIRE,NB_LIEUX_FERRES,NB_GARES_STATIONS_TRANSPORT_LOURD,NB_LIEUX_MULTIMODAUX,NB_POLES_MULTIMODAUX_LOURDS
0,75056,1298,1181,249,60,12,11,2,305,261,208,182
1,77001,4,4,0,0,0,0,0,0,0,0,0
2,77002,5,5,0,0,0,0,0,0,0,0,0
3,77003,3,3,0,0,0,0,0,0,0,0,0
4,77004,2,2,0,0,0,0,0,0,0,0,0


In [30]:
#Calculer le nombre de lignes par mode

MODES_ATTENDUS = [
    "BUS",
    "METRO",
    "TRAMWAY",
    "RER",
    "TRAIN",
    "FUNICULAIRE",
    "AUTRE",
]

lignes_par_mode = (
    arrets_lignes_idf
    .groupby(["CODGEO", "MODE"])["ROUTE_ID"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(
        columns=MODES_ATTENDUS,
        fill_value=0,
    )
    .rename(
        columns={
            "BUS": "NB_LIGNES_BUS",
            "METRO": "NB_LIGNES_METRO",
            "TRAMWAY": "NB_LIGNES_TRAMWAY",
            "RER": "NB_LIGNES_RER",
            "TRAIN": "NB_LIGNES_TRAIN",
            "FUNICULAIRE": (
                "NB_LIGNES_FUNICULAIRE"
            ),
            "AUTRE": "NB_LIGNES_AUTRES",
        }
    )
    .reset_index()
)

lignes_totales = (
    arrets_lignes_idf
    .groupby("CODGEO")
    .agg(
        NB_LIGNES_TRANSPORT=(
            "ROUTE_ID",
            "nunique",
        ),
        NB_MODES_TRANSPORT_PRESENTS=(
            "MODE",
            "nunique",
        ),
        NB_OPERATEURS_TRANSPORT=(
            "OPERATOR_NAME",
            "nunique",
        ),
    )
    .reset_index()
)

lignes_ferrees = (
    arrets_lignes_idf[
        arrets_lignes_idf["MODE"]
        .isin(MODES_FERRES)
    ]
    .groupby("CODGEO")["ROUTE_ID"]
    .nunique()
    .rename("NB_LIGNES_FERREES")
    .reset_index()
)

lignes_lourdes = (
    arrets_lignes_idf[
        arrets_lignes_idf["MODE"]
        .isin(MODES_LOURDS)
    ]
    .groupby("CODGEO")["ROUTE_ID"]
    .nunique()
    .rename("NB_LIGNES_TRANSPORT_LOURD")
    .reset_index()
)

In [37]:
#Construire la table communale de stransports

transport_communes = (
    lieux_communes
    .merge(
        lignes_totales,
        on="CODGEO",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        lignes_par_mode,
        on="CODGEO",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        lignes_ferrees,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )
    .merge(
        lignes_lourdes,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )
)

colonnes_comptage_transport = [
    colonne
    for colonne in transport_communes.columns
    if colonne != "CODGEO"
]

transport_communes[
    colonnes_comptage_transport
] = (
    transport_communes[
        colonnes_comptage_transport
    ]
    .fillna(0)
    .astype(int)
)

transport_communes["A_BUS"] = (
    transport_communes["NB_LIGNES_BUS"] > 0
).astype(int)

transport_communes["A_METRO"] = (
    transport_communes["NB_LIGNES_METRO"] > 0
).astype(int)

transport_communes["A_TRAMWAY"] = (
    transport_communes["NB_LIGNES_TRAMWAY"] > 0
).astype(int)

transport_communes["A_RER"] = (
    transport_communes["NB_LIGNES_RER"] > 0
).astype(int)

transport_communes["A_TRAIN"] = (
    transport_communes["NB_LIGNES_TRAIN"] > 0
).astype(int)

transport_communes[
    "A_TRANSPORT_LOURD"
] = (
    transport_communes[
        "NB_LIGNES_TRANSPORT_LOURD"
    ] > 0
).astype(int)

display(transport_communes.head())

,CODGEO,NB_LIEUX_TRANSPORT,NB_LIEUX_BUS,NB_STATIONS_METRO,NB_STATIONS_TRAMWAY,NB_GARES_RER,NB_GARES_TRAIN,NB_STATIONS_FUNICULAIRE,NB_LIEUX_FERRES,NB_GARES_STATIONS_TRANSPORT_LOURD,NB_LIEUX_MULTIMODAUX,NB_POLES_MULTIMODAUX_LOURDS,NB_LIGNES_TRANSPORT,NB_MODES_TRANSPORT_PRESENTS,NB_OPERATEURS_TRANSPORT,NB_LIGNES_BUS,NB_LIGNES_METRO,NB_LIGNES_TRAMWAY,NB_LIGNES_RER,NB_LIGNES_TRAIN,NB_LIGNES_FUNICULAIRE,NB_LIGNES_AUTRES,NB_LIGNES_FERREES,NB_LIGNES_TRANSPORT_LOURD,A_BUS,A_METRO,A_TRAMWAY,A_RER,A_TRAIN,A_TRANSPORT_LOURD
0,75056,1298,1181,249,60,12,11,2,305,261,208,182,251,7,33,199,19,4,6,17,1,5,47,42,1,1,1,1,1,1
1,77001,4,4,0,0,0,0,0,0,0,0,0,2,1,1,2,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,77002,5,5,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,77003,3,3,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4,77004,2,2,0,0,0,0,0,0,0,0,0,4,1,2,4,0,0,0,0,0,0,0,0,1,0,0,0,0,0


In [38]:
#Vérifier le nombre de codes communes non appariés

codes_transport = set(
    transport_communes["CODGEO"].dropna()
)

codes_non_apparies = sorted(
    codes_transport - codes_profil
)

diagnostic_codes = transport_communes[
    transport_communes["CODGEO"]
    .isin(codes_non_apparies)
].copy()

print(
    "Codes IDFM absents du profil J8 :",
    len(codes_non_apparies),
)

print(codes_non_apparies[:50])

display(diagnostic_codes.head(50))

Codes IDFM absents du profil J8 : 0
[]


,CODGEO,NB_LIEUX_TRANSPORT,NB_LIEUX_BUS,NB_STATIONS_METRO,NB_STATIONS_TRAMWAY,NB_GARES_RER,NB_GARES_TRAIN,NB_STATIONS_FUNICULAIRE,NB_LIEUX_FERRES,NB_GARES_STATIONS_TRANSPORT_LOURD,NB_LIEUX_MULTIMODAUX,NB_POLES_MULTIMODAUX_LOURDS,NB_LIGNES_TRANSPORT,NB_MODES_TRANSPORT_PRESENTS,NB_OPERATEURS_TRANSPORT,NB_LIGNES_BUS,NB_LIGNES_METRO,NB_LIGNES_TRAMWAY,NB_LIGNES_RER,NB_LIGNES_TRAIN,NB_LIGNES_FUNICULAIRE,NB_LIGNES_AUTRES,NB_LIGNES_FERREES,NB_LIGNES_TRANSPORT_LOURD,A_BUS,A_METRO,A_TRAMWAY,A_RER,A_TRAIN,A_TRANSPORT_LOURD


In [39]:
#Enregistrer les tables intermédiaires

FICHIER_ARRETS_NORMALISES = (
    DOSSIER_INTERIM
    / "j9_arrets_lignes_idfm_normalises.csv"
)

FICHIER_LIEUX_TRANSPORT = (
    DOSSIER_INTERIM
    / "j9_lieux_transport_idf.csv"
)

FICHIER_TRANSPORT_COMMUNES = (
    DOSSIER_INTERIM
    / "j9_transport_communes_idf.csv"
)

FICHIER_DIAGNOSTIC = (
    DOSSIER_INTERIM
    / "j9_diagnostic_codes_non_apparies.csv"
)

enregistrer_csv(
    arrets_lignes_idf,
    FICHIER_ARRETS_NORMALISES,
)

enregistrer_csv(
    lieux_transport,
    FICHIER_LIEUX_TRANSPORT,
)

enregistrer_csv(
    transport_communes,
    FICHIER_TRANSPORT_COMMUNES,
)

enregistrer_csv(
    diagnostic_codes,
    FICHIER_DIAGNOSTIC,
)

Enregistré : data\interim\j9_arrets_lignes_idfm_normalises.csv — 74,674 lignes
Enregistré : data\interim\j9_lieux_transport_idf.csv — 18,822 lignes
Enregistré : data\interim\j9_transport_communes_idf.csv — 1,261 lignes
Enregistré : data\interim\j9_diagnostic_codes_non_apparies.csv — 0 lignes


In [40]:
#Joindre les données IDFM au dataset initial J8

profil_j9 = profil_j8.merge(
    transport_communes,
    on="CODGEO",
    how="left",
    validate="one_to_one",
    indicator=True,
)

controle_jointure = (
    profil_j9["_merge"]
    .value_counts(dropna=False)
)

print(controle_jointure)

profil_j9 = profil_j9.drop(
    columns="_merge"
)

colonnes_transport_j9 = [
    colonne
    for colonne in transport_communes.columns
    if colonne != "CODGEO"
]

profil_j9[colonnes_transport_j9] = (
    profil_j9[colonnes_transport_j9]
    .fillna(0)
    .astype(int)
)

print("Communes avant jointure :", len(profil_j8))
print("Communes après jointure :", len(profil_j9))

assert len(profil_j9) == len(profil_j8)
assert profil_j9["CODGEO"].is_unique

print("Jointure J8 + transports réussie ✅")

_merge
both          1261
left_only        5
right_only       0
Name: count, dtype: int64
Communes avant jointure : 1266
Communes après jointure : 1266
Jointure J8 + transports réussie ✅


In [43]:
#Identifier la colonne de population

colonnes_population_possibles = [
    "POPULATION_2022"
]

COLONNE_POPULATION = next(
    (
        colonne
        for colonne in colonnes_population_possibles
        if colonne in profil_j9.columns
    ),
    None,
)

if COLONNE_POPULATION is None:
    raise ValueError(
        "Colonne de population introuvable.\n"
        f"Colonnes disponibles : "
        f"{profil_j9.columns.tolist()}"
    )

profil_j9[COLONNE_POPULATION] = (
    convertir_nombre(
        profil_j9[COLONNE_POPULATION]
    )
)

print(
    "Colonne de population utilisée :",
    COLONNE_POPULATION,
)

Colonne de population utilisée : POPULATION_2022


In [44]:
#Calculer les densités de transport

population = (
    profil_j9[COLONNE_POPULATION]
    .replace(0, np.nan)
)

profil_j9[
    "NB_LIEUX_TRANSPORT_10000_HAB"
] = (
    profil_j9["NB_LIEUX_TRANSPORT"]
    / population
    * 10_000
)

profil_j9[
    "NB_LIGNES_TRANSPORT_10000_HAB"
] = (
    profil_j9["NB_LIGNES_TRANSPORT"]
    / population
    * 10_000
)

profil_j9[
    "NB_LIGNES_FERREES_10000_HAB"
] = (
    profil_j9["NB_LIGNES_FERREES"]
    / population
    * 10_000
)

profil_j9[
    "NB_GARES_STATIONS_LOURDES_10000_HAB"
] = (
    profil_j9[
        "NB_GARES_STATIONS_TRANSPORT_LOURD"
    ]
    / population
    * 10_000
)

profil_j9[
    "NB_POLES_MULTIMODAUX_10000_HAB"
] = (
    profil_j9[
        "NB_POLES_MULTIMODAUX_LOURDS"
    ]
    / population
    * 10_000
)

In [46]:
#Calculer les indices IDF base 100

population_idf = profil_j9[
    COLONNE_POPULATION
].sum()

densite_lieux_idf = (
    profil_j9["NB_LIEUX_TRANSPORT"].sum()
    / population_idf
    * 10_000
)

densite_lignes_ferrees_idf = (
    profil_j9["NB_LIGNES_FERREES"].sum()
    / population_idf
    * 10_000
)

profil_j9[
    "INDICE_LIEUX_TRANSPORT_IDF_BASE100"
] = (
    profil_j9[
        "NB_LIEUX_TRANSPORT_10000_HAB"
    ]
    / densite_lieux_idf
    * 100
)

profil_j9[
    "INDICE_LIGNES_FERREES_IDF_BASE100"
] = (
    profil_j9[
        "NB_LIGNES_FERREES_10000_HAB"
    ]
    / densite_lignes_ferrees_idf
    * 100
)

print(
    "Densité régionale de lieux de transport :",
    round(densite_lieux_idf, 2),
    "pour 10 000 habitants",
)

print(
    "Densité régionale de lignes ferrées :",
    round(densite_lignes_ferrees_idf, 2),
    "pour 10 000 habitants",
)

Densité régionale de lieux de transport : 15.2 pour 10 000 habitants
Densité régionale de lignes ferrées : 0.44 pour 10 000 habitants


In [47]:
#Lier les transports aux emplois

if "EMPLOIS_15P_LT" in profil_j9.columns:
    profil_j9["EMPLOIS_15P_LT"] = (
        convertir_nombre(
            profil_j9["EMPLOIS_15P_LT"]
        )
    )

    emplois = (
        profil_j9["EMPLOIS_15P_LT"]
        .replace(0, np.nan)
    )

    profil_j9[
        "NB_LIGNES_TRANSPORT_1000_EMPLOIS"
    ] = (
        profil_j9["NB_LIGNES_TRANSPORT"]
        / emplois
        * 1_000
    )

    profil_j9[
        "NB_GARES_STATIONS_LOURDES_1000_EMPLOIS"
    ] = (
        profil_j9[
            "NB_GARES_STATIONS_TRANSPORT_LOURD"
        ]
        / emplois
        * 1_000
    )

    print(
        "Indicateurs rapportés aux emplois calculés ✅"
    )

else:
    print(
        "EMPLOIS_15P_LT absent : "
        "indicateurs par emploi non calculés."
    )

Indicateurs rapportés aux emplois calculés ✅


In [49]:
#Créer une typologie de desserte

conditions_desserte = [
    (
        profil_j9[
            "NB_POLES_MULTIMODAUX_LOURDS"
        ] > 0
    ),
    (
        profil_j9[
            "NB_GARES_STATIONS_TRANSPORT_LOURD"
        ] > 0
    ),
    (
        profil_j9["NB_LIEUX_FERRES"] > 0
    ),
    (
        profil_j9["NB_LIEUX_BUS"] > 0
    ),
    (
        profil_j9["NB_LIEUX_TRANSPORT"] > 0
    ),
]

classes_desserte = [
    "POLE_MULTIMODAL_LOURD",
    "TRANSPORT_LOURD",
    "DESSERTE_FERREE_LEGERE",
    "BUS_UNIQUEMENT",
    "AUTRE_DESSERTE",
]

profil_j9[
    "CLASSE_DESSERTE_TRANSPORT"
] = np.select(
    conditions_desserte,
    classes_desserte,
    default="AUCUNE_DESSERTE_REFERENCEE",
)

display(
    profil_j9[
        "CLASSE_DESSERTE_TRANSPORT"
    ]
    .value_counts(dropna=False)
    .rename_axis("CLASSE")
    .reset_index(name="NB_COMMUNES")
)

,CLASSE,NB_COMMUNES
0,BUS_UNIQUEMENT,946
1,POLE_MULTIMODAL_LOURD,165
2,TRANSPORT_LOURD,125
3,DESSERTE_FERREE_LEGERE,25
4,AUCUNE_DESSERTE_REFERENCEE,5


In [50]:
#Afficher les communes les mieux desservies

colonnes_nom_possibles = [
    "LIBELLE",
    "NOM_COMMUNE",
    "LIBGEO",
    "NOM_COM",
]

COLONNE_NOM_COMMUNE = next(
    (
        colonne
        for colonne in colonnes_nom_possibles
        if colonne in profil_j9.columns
    ),
    None,
)

colonnes_classement = [
    "CODGEO",
    COLONNE_NOM_COMMUNE,
    COLONNE_POPULATION,
    "NB_LIEUX_TRANSPORT",
    "NB_LIGNES_TRANSPORT",
    "NB_LIGNES_BUS",
    "NB_LIGNES_METRO",
    "NB_LIGNES_TRAMWAY",
    "NB_LIGNES_RER",
    "NB_LIGNES_TRAIN",
    "NB_LIGNES_TRANSPORT_LOURD",
    "NB_POLES_MULTIMODAUX_LOURDS",
    "CLASSE_DESSERTE_TRANSPORT",
]

colonnes_classement = [
    colonne
    for colonne in colonnes_classement
    if colonne is not None
    and colonne in profil_j9.columns
]

classement_desserte = (
    profil_j9
    .sort_values(
        [
            "NB_LIGNES_TRANSPORT_LOURD",
            "NB_POLES_MULTIMODAUX_LOURDS",
            "NB_LIGNES_TRANSPORT",
        ],
        ascending=False,
    )
    [colonnes_classement]
    .head(30)
)

display(classement_desserte)

,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_LIEUX_TRANSPORT,NB_LIGNES_TRANSPORT,NB_LIGNES_BUS,NB_LIGNES_METRO,NB_LIGNES_TRAMWAY,NB_LIGNES_RER,NB_LIGNES_TRAIN,NB_LIGNES_TRANSPORT_LOURD,NB_POLES_MULTIMODAUX_LOURDS,CLASSE_DESSERTE_TRANSPORT
0,75056,Paris,2113705.0,1298,251,199,19,4,6,17,42,182,POLE_MULTIMODAL_LOURD
1027,93066,Saint-Denis,148907.0,121,41,28,3,5,2,1,6,9,POLE_MULTIMODAL_LOURD
754,78646,Versailles,83918.0,143,62,53,0,2,0,6,6,2,POLE_MULTIMODAL_LOURD
962,92004,Asnières-sur-Seine,91457.0,59,31,24,2,2,1,2,5,4,POLE_MULTIMODAL_LOURD
644,78361,Mantes-la-Jolie,44246.0,64,37,33,0,0,0,4,4,0,TRANSPORT_LOURD
961,92002,Antony,64026.0,90,29,23,0,1,2,1,3,4,POLE_MULTIMODAL_LOURD
571,78172,Conflans-Sainte-Honorine,36306.0,73,27,23,0,0,1,2,3,3,POLE_MULTIMODAL_LOURD
979,92044,Levallois-Perret,68412.0,36,22,19,1,0,1,1,3,3,POLE_MULTIMODAL_LOURD
870,91377,Massy,50597.0,74,49,43,0,1,2,1,3,2,POLE_MULTIMODAL_LOURD
102,77108,Chelles,54372.0,88,39,35,0,0,2,1,3,2,POLE_MULTIMODAL_LOURD


In [51]:
#Contrôles finaux
assert len(profil_j9) == len(profil_j8)

assert profil_j9["CODGEO"].is_unique

assert (
    profil_j9["CODGEO"]
    .str.fullmatch(r"\d{5}", na=False)
    .all()
)

assert transport_communes["CODGEO"].is_unique

for colonne in colonnes_transport_j9:
    assert profil_j9[colonne].ge(0).all()

assert (
    profil_j9["NB_LIEUX_MULTIMODAUX"]
    <= profil_j9["NB_LIEUX_TRANSPORT"]
).all()

assert (
    profil_j9["NB_POLES_MULTIMODAUX_LOURDS"]
    <= profil_j9["NB_LIEUX_MULTIMODAUX"]
).all()

assert (
    profil_j9[
        "NB_GARES_STATIONS_TRANSPORT_LOURD"
    ]
    <= profil_j9["NB_LIEUX_TRANSPORT"]
).all()

assert (
    profil_j9["NB_LIGNES_TRANSPORT_LOURD"]
    <= profil_j9["NB_LIGNES_TRANSPORT"]
).all()

assert (
    profil_j9["NB_LIGNES_FERREES"]
    <= profil_j9["NB_LIGNES_TRANSPORT"]
).all()

print("Tous les contrôles J9 sont validés ✅")

Tous les contrôles J9 sont validés ✅


In [52]:
#Enregistrer le dataset profil J9 sous .csv

FICHIER_PROFIL_J9 = (
    DOSSIER_PROCESSED
    / "profil_communes_idf_j9.csv"
)

enregistrer_csv(
    profil_j9,
    FICHIER_PROFIL_J9,
)

print()
print("J9 terminé ✅")
print("Profil final :", FICHIER_PROFIL_J9)
print("Nombre de communes :", len(profil_j9))
print("Nombre de colonnes :", len(profil_j9.columns))

Enregistré : OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j9.csv — 1,266 lignes

J9 terminé ✅
Profil final : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j9.csv
Nombre de communes : 1266
Nombre de colonnes : 403
